# EmpaChat — train on a free Colab T4 and publish to the Hugging Face Hub

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

| Stage | Data | What it learns |
|---|---|---|
| 1 · Emotion classifier | GoEmotions (43K Reddit comments) | positive / negative / neutral |
| 2 · Reply generator | EmpatheticDialogues (≈19K real listener replies) | an empathetic reply, conditioned on the emotion |

Both stages fine-tune **GPT-2 XL (1.5B)** with 4-bit **QLoRA**. Each stage uploads its adapter (plus the
metrics it actually measured) to your Hub repo the moment it finishes, so the
[EmpaChat Space](https://huggingface.co/spaces/sikdaarrr/EmpaChat) can use it.

**If Colab disconnects:** re-run cells 1–4, then the stage that was interrupted — it resumes from the latest
checkpoint saved to Google Drive. Skip any stage that already printed "Uploaded to …".

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Keep checkpoints on Google Drive so a disconnect doesn't lose progress
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/empachat_outputs'
HUB_REPO = 'sikdaarrr/empachat-gpt2xl-lora'

In [ ]:
# 3. Get the code and pinned dependencies
!git clone -q https://github.com/sayantansikdar/empathetic-ai.git /content/empathetic-ai 2>/dev/null || git -C /content/empathetic-ai pull -q
%cd /content/empathetic-ai
!pip install -q transformers==5.19.0 peft==0.21.2 datasets==5.1.0 accelerate bitsandbytes scikit-learn huggingface_hub

In [ ]:
# 4. Log in to Hugging Face with a WRITE token (huggingface.co → Settings → Access Tokens)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 5. Stage 1 — emotion classifier (GoEmotions, 1 epoch)
!python train.py --lora --quantize --model gpt2-xl --epochs 1 \
    --batch_size 16 --max_length 64 --save_steps 500 \
    --skip_generator --skip_ab_test \
    --output_dir "$OUT" --hub_repo "$HUB_REPO"

In [ ]:
# 6. Stage 2 — reply generator (EmpatheticDialogues, 1 epoch)
!python train.py --lora --quantize --model gpt2-xl --epochs 1 \
    --gen_batch_size 8 --gen_max_length 128 --save_steps 500 \
    --skip_classifier --skip_ab_test \
    --output_dir "$OUT" --hub_repo "$HUB_REPO"

In [ ]:
# 7. What the run measured (also published next to each adapter on the Hub)
import json
cls = json.load(open(f'{OUT}/classifier/classifier_lora/metrics.json'))
gen = json.load(open(f'{OUT}/generator/generator_lora/metrics.json'))
print(f"Classifier: {cls['test_accuracy']*100:.1f}% test accuracy, macro-F1 {cls['test_f1_macro']:.3f} on {cls['n_test']:,} comments")
print(f"Generator:  val perplexity {gen['val_perplexity']} on {gen['n_val']:,} held-out replies ({gen['n_train']:,} training pairs)")
print(f"LoRA:       {gen['trainable_params']:,} trainable of {gen['total_params']:,} parameters")
print()
for s in json.load(open(f'{OUT}/generator/generator_lora/samples.json'))['samples']:
    print(f"[{s['emotion']}] {s['user_text']}\n  → {s['reply']}\n")
print(cls['report'])